In [76]:
import numpy as np
import random

def normalized(v):
    return v / np.linalg.norm(v)

def get_k1(e):
    up = np.array([0, 1, 1])
    t = [normalized(v) for v in e]
    d2 = [normalized(np.cross(up, v)) for v in e]
    X = 1 + np.dot(t[0], t[1])
    kb = 2 * np.cross(t[0], t[1]) / X
    k1 = 0.5 * np.dot(d2[0] + d2[1], kb)

    c = np.cross(t[1], sum(d2))

    dk1de0 = (-k1 * sum(t) + np.cross(t[1], sum(d2))) / np.linalg.norm(e[0]) / X
    # dk1de0 = (-t[0] * t[0].dot(c) + np.cross(t[1], sum(d2))) / np.linalg.norm(e[0]) / X
    # dk1de0 = (np.cross(t[1], sum(d2))) / np.linalg.norm(e[0]) / X
    return k1, dk1de0

e = []
e.append([1, 0, 0])
e.append(normalized([1, 0.1, 0]))
# e.append([1, 0, 0])
e = [np.array(v, dtype=np.float64) for v in e]

k1, dk1 = get_k1(e)
for i in range(3):
    offset = np.array([1 if i == j else 0 for j in range(3)], dtype=np.float64)
    stepsize = 1e-8
    offset = offset * stepsize
    e_new = e.copy()
    e_new[0] += offset
    k1_new, _ = get_k1(e_new)
    fd = k1_new - k1
    taylor = offset.dot(dk1)
    err = (fd - taylor) / (abs(fd) + stepsize)
    print(err, fd, taylor)

display(k1, dk1)




1.3912309881655627e-17 0.0 -1.3912309881655628e-25
-5.2809021321826525e-09 7.0798298196228515e-09 7.079829909819761e-09
0.2930429086454956 1.4159703354277298e-08 7.079873611327542e-09


-0.07044694060631695

array([-1.39123099e-17,  7.07982991e-01,  7.07987361e-01])

In [140]:
import importlib
from toy import sympy_pearlmutter as sp
import sympy
importlib.reload(sp)

e0 = sympy.Symbol(r'e^{i-1}')
e1 = sympy.Symbol(r'e^{i}')
k1 = sympy.Symbol(r'\kappa_1')
chi = sympy.Symbol(r'\Chi')
t0 = e0 / sp.Norm(e0)
t1 = e1 / sp.Norm(e1)
t_tilde = (t0 + t1) / chi
d2_tilde = sympy.Symbol(r'\tilde{d_2}')
print(isinstance(e0, sympy.Symbol))

force = 1 / sp.Norm(e0) * (-k1 * t_tilde + sp.Cross(t1, d2_tilde))
def out(exp):
    exp = exp.replace(t0, sympy.Symbol(r't^{i - 1}'))
    exp = exp.replace(t1, sympy.Symbol(r't^{i}'))
    # exp.replace(lambda x: isinstance(x, sympy.Symbol) and x != e0 and x != e1, lambda x: 0)
    exp = exp.replace(sympy.Symbol(r'\partial X'), 0)
    exp = exp.replace(sympy.Symbol(r'\partial \tilde{d_2}'), 0)
    exp = exp.replace(sympy.Symbol(r'\partial \kappa_1'), 0)
    display(exp)
# force = 1 / sp.Norm(e0)
display(force)
out(force)
hessian = sp.get_partial(force)
display(hessian)
out(hessian)

True


(Cross(e^{i}/Norm(e^{i}), \tilde{d_2}) - \kappa_1*(e^{i-1}/Norm(e^{i-1}) + e^{i}/Norm(e^{i}))/\Chi)/Norm(e^{i-1})

(Cross(t^{i}, \tilde{d_2}) - \kappa_1*(t^{i - 1} + t^{i})/\Chi)/Norm(e^{i-1})

-(Cross(e^{i}/Norm(e^{i}), \tilde{d_2}) - \kappa_1*(e^{i-1}/Norm(e^{i-1}) + e^{i}/Norm(e^{i}))/\Chi)*Dot(e^{i-1}, \partial e^{i-1})/Norm(e^{i-1})**3 + (Cross(e^{i}/Norm(e^{i}), \partial \tilde{d_2}) + Cross(\partial e^{i}/Norm(e^{i}) - e^{i}*Dot(e^{i}, \partial e^{i})/Norm(e^{i})**3, \tilde{d_2}) - \kappa_1*(\partial e^{i-1}/Norm(e^{i-1}) + \partial e^{i}/Norm(e^{i}) - e^{i-1}*Dot(e^{i-1}, \partial e^{i-1})/Norm(e^{i-1})**3 - e^{i}*Dot(e^{i}, \partial e^{i})/Norm(e^{i})**3)/\Chi - \partial \kappa_1*(e^{i-1}/Norm(e^{i-1}) + e^{i}/Norm(e^{i}))/\Chi + \kappa_1*\partial \Chi*(e^{i-1}/Norm(e^{i-1}) + e^{i}/Norm(e^{i}))/\Chi**2)/Norm(e^{i-1})

-(Cross(t^{i}, \tilde{d_2}) - \kappa_1*(t^{i - 1} + t^{i})/\Chi)*Dot(e^{i-1}, \partial e^{i-1})/Norm(e^{i-1})**3 + (Cross(\partial e^{i}/Norm(e^{i}) - e^{i}*Dot(e^{i}, \partial e^{i})/Norm(e^{i})**3, \tilde{d_2}) - \kappa_1*(\partial e^{i-1}/Norm(e^{i-1}) + \partial e^{i}/Norm(e^{i}) - e^{i-1}*Dot(e^{i-1}, \partial e^{i-1})/Norm(e^{i-1})**3 - e^{i}*Dot(e^{i}, \partial e^{i})/Norm(e^{i})**3)/\Chi + \kappa_1*\partial \Chi*(t^{i - 1} + t^{i})/\Chi**2)/Norm(e^{i-1})